[← GstreamerExp hub](../../index.html) · [README](../../README.md) · [Hypothesis catalog](../../docs/HYPOTHESES.md)

# H6 — SCReAM's queue-delay-target moves latency but not picture quality

**Status:** `refuted` · **Source:** Goal 2.2 (knob sensitivity), project-internal


## Claim

Turning SCReAM's queue-delay-target changes p95 latency on every network we tested, but does not change decoded picture quality (PSNR) — at this workload there is no quality gain to trade for the added latency.

## Predictions

- `psnr_rises_with_knob_on_every_network`
- `p95_latency_rises_with_knob_on_every_network`
- `diminishing_returns_point_visible_on_every_network`
- `diminishing_returns_points_differ_between_networks`

## Verdict

| Outcome | Predicate |
|---|---|
| **Supported when all** | <code>psnr_rises_with_knob_on_every_network</code><br><code>p95_latency_rises_with_knob_on_every_network</code><br><code>diminishing_returns_point_visible_on_every_network</code><br><code>diminishing_returns_points_differ_between_networks</code> |
| **Refuted when any** | <code>psnr_does_not_rise_with_knob_on_at_least_one_network</code><br><code>no_diminishing_returns_point_visible_anywhere</code><br><code>diminishing_returns_points_match_across_networks</code> |
| **Untested when any** | <code>any_cell_failed</code><br><code>required_metric_missing</code> |


## Findings and Limitations

**Findings**

- The control loop responds the right way to the knob. As the knob goes from 10 ms to 500 ms on every network, p95 latency moves with it — tighter knob, less latency; looser knob, more latency. SCReAM is doing what its specification says it should.
- The encoder does not benefit from extra room at this loose ceiling. Decoded picture quality (PSNR) stays roughly flat and high across the knob range — about 37.7 to 39.5 dB on the steady network, 35.6 to 38.9 dB on the wobbly network, 31.8 to 34.1 dB on the 5G recording. The only consistent trend is a slight DROP at the loosest setting (500 ms), so loosening the knob never raises quality. VP8 at this 4000 kbps ceiling already has enough budget to encode this clip; extra buffering does not give it useful extra room.
- The quality numbers are trustworthy. The decoded_psnr metric was fixed before this run (frame-accurate self-healing alignment); these ~32 to 39 dB values are believable for the clip and ceiling, and the metric separates this loose ceiling from the tight 800 kbps ceiling in H7 (19 to 34 dB). The earlier broken metric pinned everything near 10 dB and could not.

**Limitations**

- Workload-specific. Tested with realmotion-avi (1280×1024 MJPEG, 10 fps, 60 s). A higher-motion or longer clip might saturate the encoder differently.
- Bitrate-specific. Tested at init / min / max = 1500 / 200 / 4000 kbps. H7 re-ran the same sweep at a tight 800 kbps ceiling; the knob still did not raise quality there either.
- The "no quality response" claim rests on PSNR. SSIM or VMAF might surface a smaller effect that PSNR misses.
- 3 reps per cell; effects smaller than the run-to-run spread are not resolved.
- Absolute latency numbers carry a clock-skew artifact between aum and veda (same one H4 flagged). The relative comparison across knob values within one sweep is unaffected.


## Experimental setup


## Required metrics

- `frame_count`
- `encoder_target_kbps`
- `wire_bytes`
- `encoded_bitrate`
- `frame_latency`
- `decoder_errors`
- `decoded_psnr`


## Reproducibility

This notebook is generated from `specs/hypotheses/h6.yaml` and `analysis/hypotheses/results/h6_report.json`. To regenerate:

```sh
python3 analysis/hypotheses/build_reports.py
python3 analysis/hypotheses/build_pages.py
python3 analysis/hypotheses/h6_qdt_sweep.py
```

Source: Goal 2.2 (knob sensitivity), project-internal
